In [2]:
#Force an update to the latest compatible versions
!pip install --upgrade transformers huggingface_hub accelerate datasets
print("Libraries updated! Now go to 'Runtime' -> 'Restart Session' at the top.") #confirmation message to know status

  Using cached huggingface_hub-1.2.3-py3-none-any.whl.metadata (13 kB)
Libraries updated! Now go to 'Runtime' -> 'Restart Session' at the top.


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

print("Loading XLCoST with the correct configuration...") #confirmaation messages to know the state

# Using 'Java-program-level' as the source
# The error showed these are the names the dataset recognizes
dataset = load_dataset("codeparrot/xlcost-text-to-code", "Java-program-level", trust_remote_code=True)
print("Dataset Loaded!") #confirmaation messages to know the state

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")

def preprocess_function(examples):
    # XLCoST stores code in the 'code' column
    # We will use the Java code as input and we need to find the matching Python
    # However, to keep it simple and ensure we have pairs, we'll use this mapping:
    inputs = ["translate Java to Python: " + code for code in examples["code"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # We will use the same examples for labels for now to test the pipeline
    # In a full run, we would map to the Python-program-level split
    labels = tokenizer(examples["code"], max_length=512, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Preprocessing data...") #confirmaation messages to know the state
tokenized_dataset = dataset.map(preprocess_function, batched=True)
print("Data is ready!") #confirmaation messages to know the state

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'codeparrot/xlcost-text-to-code' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'codeparrot/xlcost-text-to-code' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading XLCoST with the correct configuration...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using the latest cached version of the dataset since codeparrot/xlcost-text-to-code couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'Java-program-level' at /root/.cache/huggingface/datasets/codeparrot___xlcost-text-to-code/Java-program-level/2.1.0/60c5c133f043a5cffe162f9de1c62b9d88f309cf (last modified on Tue Dec 16 13:54:21 2025).


Dataset Loaded!
Preprocessing data...


Map:   0%|          | 0/494 [00:00<?, ? examples/s]

Data is ready!


In [4]:
from transformers import T5ForConditionalGeneration, Trainer, TrainingArguments

# Load the base model (the 'untrained brain')
model = T5ForConditionalGeneration.from_pretrained("Salesforce/codet5-base")

# Set the rules for training
training_args = TrainingArguments(
    output_dir="./codet5-java-to-python",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # Updates every 2 steps to keep the math the same
    num_train_epochs=3,
    save_total_limit=1,
    fp16=True,
    report_to="none"
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
)

# START TRAINING
print("Starting training... Look for the progress bar below!") #confirmaation messages to know the state
trainer.train()

# Save the finished model
model.save_pretrained("./codet5-java-to-python")
tokenizer.save_pretrained("./codet5-java-to-python")
print("Model Saved Successfully!") #confirmaation messages to know the state

Starting training... Look for the progress bar below!


Epoch,Training Loss,Validation Loss
1,0.004900,0.000884
2,0.002700,0.000758
3,0.001800,0.000654


Model Saved Successfully!


In [6]:
import shutil
from google.colab import files

# Create a zip file of the model folder
shutil.make_archive("fine_tuned_codet5", 'zip', "./codet5-java-to-python")

# Download it to your computer
files.download("fine_tuned_codet5.zip")
print("Downloading your model... Check your 'Downloads' folder!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>